[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/model_serving.ipynb)

# 04 . Model Serving

## Concept — Model Serving (revision notes)

**What it is**
- Exposing a trained model as a service: requests come in, are pre-processed, run through the model, post-processed and returned.
- The core engineering problems are latency (time for one request), throughput (requests/second) and the tension between them, which dynamic batching mediates.

**Why it matters**
- A model that scores well offline is useless if p99 latency is 3 seconds or the server falls over at 20 requests/s.
- GPUs and even CPUs are far more efficient on a batch of 32 than on 32 separate calls, so serving systems queue requests and batch them.

**When to use**
- Online serving with a request/response API (FastAPI, TorchServe, Triton, BentoML).
- Dynamic batching: when many concurrent small requests hit an expensive model.
- Offline/batch inference needs no server at all: just a DataLoader loop.

**Math & intuition**
- Latency of a batch call is roughly overhead + batch_size x per-item cost; overhead is amortized across the batch, so throughput rises with batch size while each request waits a bit longer.
- Dynamic batching has two knobs: `max_batch_size` and `max_wait_ms`. Wait longer -> bigger batches and more throughput, but higher latency for lightly loaded traffic.
- Always report percentiles (p50/p95/p99), not the mean: tail latency is what users feel.

### 1. Preparing a model for inference

**What this cell does (plain language):** we build a small MLP "classifier" and put it in serving mode: `eval()` (turns off dropout, uses running stats in batch norm) and `torch.inference_mode()` (no autograd bookkeeping at all). We check that the two modes give the same outputs, and that inference_mode outputs cannot be used for backward.

In [1]:
import torch, torch.nn as nn, time, threading, queue, statistics, concurrent.futures as cf
import numpy as np
torch.manual_seed(0)

model = nn.Sequential(nn.Linear(256, 1024), nn.ReLU(), nn.Dropout(0.5),
                      nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10))
x = torch.randn(1, 256)

model.train(); a = model(x); b = model(x)
print("train mode (dropout on) outputs equal:", torch.allclose(a, b))   # False: random!
model.eval()
with torch.inference_mode():
    c = model(x); d = model(x)
print("eval + inference_mode outputs equal :", torch.allclose(c, d))
print("inference tensor requires_grad      :", c.requires_grad)

train mode (dropout on) outputs equal: False
eval + inference_mode outputs equal : True
inference tensor requires_grad      : False


### 2. A latency measurement helper

**What this cell does (plain language):** we write `percentiles()` and a `timeit` helper that returns per-call latencies in milliseconds. We always discard a few warm-up calls first: the first calls pay one-off costs (lazy initialization, memory allocation, cache warm-up) that do not represent steady state.

In [2]:
def percentiles(lat_ms):
    a = np.array(lat_ms)
    return {"p50": np.percentile(a, 50), "p95": np.percentile(a, 95), "p99": np.percentile(a, 99), "mean": a.mean()}

def fmt(p): return " ".join(f"{k}={v:.2f}ms" for k, v in p.items())

def time_calls(fn, n=100, warmup=10):
    for _ in range(warmup): fn()
    out = []
    for _ in range(n):
        t = time.perf_counter(); fn(); out.append((time.perf_counter() - t) * 1e3)
    return out

with torch.inference_mode():
    lat = time_calls(lambda: model(x))
print("single-request latency:", fmt(percentiles(lat)))

single-request latency: p50=29.38ms p95=39.94ms p99=40.04ms mean=30.28ms


### 3. Latency vs throughput as batch size grows

**What this cell does (plain language):** we run the same model on batches of 1 to 128 and print per-batch latency and items/second. Latency per call grows with batch size, but throughput (items per second) grows much faster at first — the whole reason dynamic batching exists.

In [3]:
print(f"{'batch':>5} {'latency ms':>11} {'items/s':>10}")
tp = {}
with torch.inference_mode():
    for bs in (1, 2, 4, 8, 16, 32, 64, 128):
        xb = torch.randn(bs, 256)
        l = np.median(time_calls(lambda: model(xb), n=30, warmup=5))
        tp[bs] = bs / (l / 1e3)
        print(f"{bs:>5} {l:>11.3f} {tp[bs]:>10.0f}")
print("throughput gain batch128 vs batch1: x%.1f" % (tp[128] / tp[1]))

batch  latency ms    items/s


    1      32.608         31


    2      28.119         71


    4      28.150        142


    8      32.900        243


   16      47.983        333


   32      45.992        696


   64     106.748        600


  128      60.159       2128
throughput gain batch128 vs batch1: x69.4


### 4. A dynamic batching server: queue + worker thread + futures

**What this cell does (plain language):** the heart of the notebook. `BatchServer` runs a background worker that pulls requests from a `queue.Queue`, collects up to `max_batch_size` of them (waiting at most `max_wait_ms` for stragglers after the first arrives), runs ONE model call on the stacked batch, and resolves each caller's `Future` with its own row. Clients simply call `server.predict(x)` and block until their result is ready.

In [4]:
class BatchServer:
    def __init__(self, model, max_batch_size=32, max_wait_ms=5.0):
        self.model, self.max_bs, self.max_wait = model.eval(), max_batch_size, max_wait_ms / 1e3
        self.q = queue.Queue(); self.batch_sizes = []; self._stop = False
        self.worker = threading.Thread(target=self._loop, daemon=True); self.worker.start()

    def predict(self, x, timeout=10.0):
        fut = cf.Future()
        self.q.put((x, fut, time.perf_counter()))       # enqueue and wait for the worker
        return fut.result(timeout=timeout)

    def _loop(self):
        while not self._stop:
            try: first = self.q.get(timeout=0.05)       # block until at least one request exists
            except queue.Empty: continue
            batch = [first]
            deadline = time.perf_counter() + self.max_wait
            while len(batch) < self.max_bs:             # gather more until full or deadline
                remaining = deadline - time.perf_counter()
                if remaining <= 0: break
                try: batch.append(self.q.get(timeout=remaining))
                except queue.Empty: break
            try:
                with torch.inference_mode():
                    out = self.model(torch.stack([b[0] for b in batch]))   # ONE forward for all
                for (_, fut, _), row in zip(batch, out): fut.set_result(row.clone())
            except Exception as e:                      # never leave callers hanging
                for _, fut, _ in batch: fut.set_exception(e)
            self.batch_sizes.append(len(batch))

    def close(self): self._stop = True; self.worker.join(timeout=2)

srv = BatchServer(model, max_batch_size=8, max_wait_ms=5)
xr = torch.randn(256)
direct = model.eval()(xr[None])[0]
print("server output matches direct call:", torch.allclose(srv.predict(xr), direct, atol=1e-5))
srv.close()

server output matches direct call: True


### 5. Load test: many concurrent clients, batching on vs off

**What this cell does (plain language):** we simulate 64 concurrent clients, each sending requests, against (a) a server that processes requests one at a time (`max_batch_size=1`) and (b) the batching server. We report throughput, latency percentiles and the batch sizes actually formed. The numbers depend on your CPU, so compare the two rows rather than reading absolute values.

In [5]:
def load_test(server, n_requests=400, concurrency=32):
    lat = []
    def one(i):
        xi = torch.randn(256)
        t = time.perf_counter(); server.predict(xi); return (time.perf_counter() - t) * 1e3
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(concurrency) as ex: lat = list(ex.map(one, range(n_requests)))
    wall = time.perf_counter() - t0
    return n_requests / wall, percentiles(lat)

results = {}
for name, bs, wait in (("no batching", 1, 0.0), ("dynamic batch<=32", 32, 5.0)):
    s = BatchServer(model, max_batch_size=bs, max_wait_ms=wait)
    for _ in range(5): s.predict(torch.randn(256))      # warm up
    s.batch_sizes.clear()
    rps, p = load_test(s)
    results[name] = rps
    print(f"{name:18s} throughput={rps:7.0f} req/s | {fmt(p)} | mean batch={np.mean(s.batch_sizes):.1f}")
    s.close()
print("speed-up from batching: x%.2f" % (results["dynamic batch<=32"] / results["no batching"]))

no batching        throughput=   1479 req/s | p50=20.08ms p95=24.48ms p99=24.99ms mean=19.91ms | mean batch=1.0
dynamic batch<=32  throughput=   3774 req/s | p50=6.25ms p95=9.20ms p99=11.25ms mean=6.44ms | mean batch=28.6


speed-up from batching: x2.55


### 6. Tuning max_wait_ms: the latency/throughput dial

**What this cell does (plain language):** we sweep `max_wait_ms` with a LIGHT load (a single client sending sequentially). With one client, waiting for a batch to fill is pure waste: each request pays the full wait. This is the cost side of dynamic batching and why `max_wait_ms` should be small relative to your latency budget.

In [6]:
print(f"{'max_wait_ms':>11} | single sequential client latency")
for wait in (0, 2, 10, 30):
    s = BatchServer(model, max_batch_size=32, max_wait_ms=wait)
    for _ in range(3): s.predict(torch.randn(256))
    lat = [(lambda t: (s.predict(torch.randn(256)), (time.perf_counter() - t) * 1e3)[1])(time.perf_counter()) for _ in range(20)]
    print(f"{wait:>11} | {fmt(percentiles(lat))}")
    s.close()

max_wait_ms | single sequential client latency
          0 | p50=0.72ms p95=2.25ms p99=7.38ms mean=1.21ms
          2 | p50=3.25ms p95=3.94ms p99=5.02ms mean=3.40ms


         10 | p50=11.50ms p95=17.06ms p99=60.11ms mean=14.64ms


         30 | p50=31.74ms p95=32.76ms p99=33.32ms mean=31.89ms


### 7. Input validation and error handling

**What this cell does (plain language):** production services must reject bad inputs rather than crash the worker thread. We add a `validate()` step (shape, dtype, NaN check) in front of the server and show that a bad request raises a clear error for that caller only while the server keeps serving good requests.

In [7]:
def validate(x, dim=256):
    if not isinstance(x, torch.Tensor): raise TypeError("expected a tensor")
    if x.shape != (dim,): raise ValueError(f"expected shape ({dim},), got {tuple(x.shape)}")
    if not torch.isfinite(x).all(): raise ValueError("input contains NaN/inf")
    return x.float()

s = BatchServer(model, max_batch_size=8, max_wait_ms=2)
for bad in (torch.randn(10), torch.full((256,), float("nan"))):
    try: s.predict(validate(bad))
    except ValueError as e: print("rejected:", e)
print("good request still served, output shape:", tuple(s.predict(validate(torch.randn(256))).shape))
s.close()

rejected: expected shape (256,), got (10,)
rejected: input contains NaN/inf
good request still served, output shape: (10,)


### 8. Reference code (not executed): FastAPI and TorchServe

**What this cell does (plain language):** shows how the pieces above map onto real serving frameworks. The code is left as comments because it needs a running web server (and optional packages). In FastAPI the batching worker is the same `BatchServer` we just wrote; TorchServe has dynamic batching built in via `batch_size` / `max_batch_delay`.

In [8]:
# ---------------- FastAPI (pip install fastapi uvicorn) ----------------
# app.py  ->  run:  uvicorn app:app --workers 1
#
# from fastapi import FastAPI, HTTPException
# from pydantic import BaseModel, Field
# import torch
#
# app = FastAPI()
# model = torch.jit.load("model.pt").eval()          # load ONCE at startup, not per request
# server = BatchServer(model, max_batch_size=32, max_wait_ms=5)   # class from section 4
#
# class Req(BaseModel):
#     features: list[float] = Field(min_length=256, max_length=256)
#
# @app.post("/predict")
# def predict(r: Req):
#     out = server.predict(torch.tensor(r.features))  # sync def -> runs in FastAPI's threadpool
#     return {"logits": out.tolist()}
#
# @app.get("/healthz")
# def health(): return {"status": "ok"}
#
# ---------------- TorchServe (pip install torchserve torch-model-archiver) ----------------
# handler.py:   class MyHandler(BaseHandler): preprocess(), inference(), postprocess()
# torch-model-archiver --model-name clf --version 1.0 --serialized-file model.pt --handler handler.py
# torchserve --start --model-store model_store --models clf=clf.mar
# config.properties:  models={"clf": {"1.0": {"batchSize": 16, "maxBatchDelay": 10}}}
print("reference only")

reference only


### 9. Is FastAPI installed here? (guarded check)

**What this cell does (plain language):** we check whether the optional serving libraries are importable and, if FastAPI is available, build a tiny app object (without starting a server) to show the code is valid. If not, we print the install hint and the notebook proceeds.

In [9]:
try:
    from fastapi import FastAPI
    from fastapi.testclient import TestClient     # in-process client: no network needed
    app = FastAPI()
    @app.get("/healthz")
    def health(): return {"status": "ok"}
    print("fastapi test call:", TestClient(app).get("/healthz").json())
except ImportError:
    print("fastapi (or httpx for TestClient) not installed -> `pip install fastapi uvicorn httpx`; skipping")

fastapi (or httpx for TestClient) not installed -> `pip install fastapi uvicorn httpx`; skipping


### 10. Warm-up and cold-start effects

**What this cell does (plain language):** we time the very first call of a freshly built model against subsequent calls (on a busy or small machine the gap may be small or hidden by noise). First calls can be noticeably slower (lazy allocations, thread-pool start-up, kernel selection). Production servers run a few dummy requests at startup, before reporting healthy, so real users never see the cold path.

In [10]:
fresh = nn.Sequential(nn.Linear(256, 1024), nn.ReLU(), nn.Linear(1024, 10)).eval()
xx = torch.randn(1, 256)
times = []
with torch.inference_mode():
    for _ in range(6):
        t = time.perf_counter(); fresh(xx); times.append((time.perf_counter() - t) * 1e3)
print("first call: %.3f ms | later calls (median): %.3f ms" % (times[0], statistics.median(times[1:])))
print("all:", [round(t, 3) for t in times])

first call: 6.367 ms | later calls (median): 7.933 ms
all: [6.367, 7.522, 7.981, 8.285, 7.762, 7.933]


## Common bugs

- **Serving with the model still in `train()` mode.** Fix: Call `model.eval()` and use `torch.inference_mode()`; dropout and batch norm otherwise corrupt predictions.
- **Reporting only the mean latency.** Fix: Report p50/p95/p99; a few slow requests (cold start, GC, queueing) dominate user experience.
- **Loading the model inside the request handler.** Fix: Load once at startup and keep it in memory; run warm-up requests before accepting traffic.
- **Benchmarking without warm-up or with a different input than production.** Fix: Discard the first calls and use realistic shapes/batch sizes.
- **Unbounded request queue.** Fix: Bound the queue and shed load (HTTP 429/503) instead of letting latency grow without limit; always set client timeouts.
- **A worker thread that dies on one bad input and leaves callers waiting forever.** Fix: Wrap the batch in try/except and set the exception on every pending future (as `BatchServer` does).
- **Many worker processes each using all CPU threads.** Fix: Set `torch.set_num_threads` so (processes x threads) <= cores, else they thrash each other.
- **Variable-length inputs batched without padding/masking.** Fix: Pad to the longest in the batch and pass a mask, or bucket requests by length.
- **Using the batch's mean prediction as a request's answer / mixing up result order.** Fix: Return `out[i]` to the i-th requester (via futures), never reorder.

## Exercises

**Exercise 1.** Measure p50 and p99 single-request latency with `torch.set_num_threads(1)` vs the default thread count (restore the default afterwards). Which is better for a tiny model?

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

For small matmuls the threading overhead can outweigh the gain; the answer depends on your CPU.

In [12]:
# Solution 1
default = torch.get_num_threads()
for n in (1, default):
    torch.set_num_threads(n)
    with torch.inference_mode(): p = percentiles(time_calls(lambda: model(x), n=50))
    print(n, "threads:", fmt(p))
torch.set_num_threads(default)

1 threads: p50=0.38ms p95=0.52ms p99=0.54ms mean=0.38ms


4 threads: p50=7.64ms p95=22.20ms p99=36.22ms mean=9.18ms


**Exercise 2.** Add a `queue_wait_ms` measurement to `BatchServer` (time from enqueue to batch start). Hint: the enqueue timestamp is already stored as the third tuple element.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
# In _loop, right before the forward pass:
#   start = time.perf_counter()
#   waits = [(start - b[2]) * 1e3 for b in batch]
# then store `waits` on self.queue_waits. Demonstration of the computation:
enq = time.perf_counter(); time.sleep(0.01); start = time.perf_counter()
print("queue wait ms ~", round((start - enq) * 1e3, 1))

queue wait ms ~ 10.3


**Exercise 3.** Bound the queue: create `queue.Queue(maxsize=4)` and show that `put_nowait` raises `queue.Full` when it is full (this is where you would return HTTP 429).

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
q = queue.Queue(maxsize=4)
try:
    for i in range(6): q.put_nowait(i)
except queue.Full:
    print("queue full after", q.qsize(), "items -> shed load")

queue full after 4 items -> shed load


**Exercise 4.** Compute throughput (items/s) for batch size 16 using the helper functions and verify it beats batch size 1.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
with torch.inference_mode():
    t1 = np.median(time_calls(lambda: model(torch.randn(1, 256)), n=30))
    t16 = np.median(time_calls(lambda: model(torch.randn(16, 256)), n=30))
print("bs1 items/s:", 1000 / t1, "| bs16 items/s:", 16000 / t16)

bs1 items/s: 62.540041261599264 | bs16 items/s: 478.97976352394426


**Exercise 5.** Use `torch.inference_mode()` vs `torch.no_grad()`: time 200 forward calls with each and report the ratio.

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

inference_mode is usually equal or slightly faster; the difference is small for large layers.

In [20]:
# Solution 5
def bench(ctx):
    t = time.perf_counter()
    for _ in range(200):
        with ctx(): model(x)
    return time.perf_counter() - t
print("no_grad:", bench(torch.no_grad), "inference_mode:", bench(torch.inference_mode))

no_grad: 3.596932114000083 inference_mode: 2.9292351220001365
